# HAM10000 ResNet Fine-tuning
This notebook demonstrates how to fine-tune a ResNet model on the HAM10000 dataset using the `ResNetFineTuneTrainer` class.

In [ ]:
import sys
import os
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

In [ ]:
from dataset_manager import DatasetManager
from pipeline.resnet_finetune_train import ResNetFineTuneTrainer

In [ ]:
# 1. Prepare the Data
# Initialize the dataset manager
data_dir = str(proj_dir / "data")
manager = DatasetManager(data_dir=data_dir)

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True, sample_size=2226)

# Split the data into train (70%), validation (15%), and test (15%) sets
train_df, val_df, test_df = manager.train_test_val_split(val_size=0.15, test_size=0.15)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 1558
Validation samples: 334
Test samples: 334


In [5]:
# 2. Configure the Trainer
trainer = ResNetFineTuneTrainer()

# Set up the model (ResNet50, 2 classes: nv vs mel)
trainer.set_model(model_name='resnet50', pretrained=True) # , input_channels=6

# Set data loaders
trainer.set_data_loaders(train_df, val_df, test_df, filter_name='lbp_filter', data_path=data_dir)

# Configure optimizer and scheduler
trainer.configure_optimizer(optimizer_type='adam', lr=1e-4, weight_decay=1e-4)
trainer.configure_scheduler(scheduler_type='reduce_on_plateau', patience=3)

# Set output paths for artifacts
trainer.set_output_paths(
    model_save_dir="../results/models",
    results_csv_path="../results/training_results.csv",
    history_plot_dir="../results/plots"
)

In [ ]:
# 3. Train the Model
# Train for a specified number of epochs
trainer.train(epochs=15)

In [14]:
# 4. Evaluate and Save Results
trainer.evaluate()
trainer.save_results_to_csv()

Evaluation results - F1 (macro): 0.7482, F1 (weighted): 0.7482, AUC: 0.8207, Accuracy: 0.7485, Sensitivity: 0.7844, Specificity: 0.7126
